# Similitud semántica

Vamos a ilustrar en qué consiste la similitud semántica con un pequeño ejemplo de buscador que utiliza embeddings.

Para ello, vamos a inspirarnos es este [ejemplo de buscador semántico en 3 minutos](https://github.com/hanxiao/bert-as-service#building-a-qa-semantic-search-engine-in-3-minutes), del proyecto [`bert-as-service`](https://github.com/hanxiao/bert-as-service) de [Han Xiao](https://hanxiao.io/), y simular que estamos construyendo un motor de búsqueda semántico sencillo.

In [1]:
import numpy as np
import spacy

from typing import List

nlp = spacy.load("en_core_web_md")

Inventamos unos cuantos documentos (en este caso, simples oraciones) de ejemplo, con poco solapamiento entre ellas. Construimos los embeddings de las oraciones utilizando los vectores de GloVe incluidos en el modelo de inglés disponible con spaCy.

In [2]:
docs = [
    "I hate tennis.",
    "I love ice cream",
    "I love riding my bicycle",
    "My car is a blue Hyunday",
    "This session is about natural language processing and artificial intelligence"
]

doc_vecs = [nlp(doc).vector for doc in docs]

Definimos una función que toma como argumentos de entrada un texto como query, el número de resultados que queremos mostrar, una colección de documentos en formato texto, y la misma colección de documentos vectorizada.

La función procesa la query de entrada con el modelo de spaCy, construye el embedding y calcula el producto escalar con respecto a todos y cada uno de los embeddings de la colección de documentos. Después, imprime los resultados más similares.

In [3]:
def search_similar_query(
    text: str, 
    topk: int,
    docs: List[str], 
    doc_vecs: List[np.ndarray]
):
    """Search for similar vectors.
    It computes the dot product between a vector and a set of vectors"""
    query = nlp(text)
    # normalized dot product as score
    score = np.sum(query.vector * doc_vecs, axis=1) / np.linalg.norm(doc_vecs, axis=1)
    topk_idx = np.argsort(score)[::-1][:topk]
    for idx in topk_idx:
        print(f"[{score[idx]}]: {docs[idx]}")

In [4]:
search_similar_query("I don't like racket sports.", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.677881956100464]: I hate tennis.
[3.0421366691589355]: I love riding my bicycle
[2.764522075653076]: I love ice cream


In [5]:
search_similar_query("I own a Toyota vehicle.", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.205249547958374]: My car is a blue Hyunday
[2.78167986869812]: I love riding my bicycle
[2.6346259117126465]: I hate tennis.


In [6]:
search_similar_query("I enjoy desserts.", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.4192099571228027]: I love ice cream
[3.2487540245056152]: I hate tennis.
[3.0283806324005127]: I love riding my bicycle


In [7]:
search_similar_query("We are talking about deep learning.", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.0805301666259766]: This session is about natural language processing and artificial intelligence
[2.647329330444336]: I hate tennis.
[2.543954610824585]: My car is a blue Hyunday


In [8]:
search_similar_query("I have too many bikes", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.5638136863708496]: I love riding my bicycle
[3.335101366043091]: I hate tennis.
[2.9381661415100098]: My car is a blue Hyunday


## Corpus de Chatbots

A continuación vamos a repetir el proceso con una colección de mensajes más grande. En el directorio `data/` del repo he incluido un par de ficheros CSV que continene datos de entrenamiento para un chatbot en inglés y en español. Vamos a cargar los datos del dataset en inglés.

In [9]:
import pandas as pd

In [10]:
df_en = pd.read_csv("data/chatbot-en.csv")
df_en[df_en["dataset"]=="training"].sample(10)

,utterance,intent,dataset
2,what is your company creating,support.about,training
80,when is your birthday,smalltalk.birthday,training
232,what is implicit coercion,trivia.coercion,training
244,why i need to know about garbage collection,trivia.gc,training
111,how boring you are,smalltalk.boring,training
172,want to hug,smalltalk.hug,training
186,it's a joke,smalltalk.joking,training
169,you can talk to me,smalltalk.talk,training
243,tell me about the garbage collector in javascript,trivia.gc,training
31,I want your creator mail,support.email,training


En nuestro caso, solo nos interesan los textos de ejemplo que aparecen en la columna *utterance*. Vamos a crear los embeddings de GloVe para los datos de entrenamiento de este dataset.

In [11]:
docs = list(df_en[df_en["dataset"]=="training"]["utterance"])
doc_vecs = [nlp(doc).vector for doc in docs]

Y vamos a probar qué tal funciona.

In [12]:
search_similar_query("You're disgusting", topk=3, docs=docs, doc_vecs=doc_vecs)

[4.194239616394043]: you're bad
[4.122173309326172]: you're so annoying
[4.074212551116943]: you're very annoying


In [13]:
search_similar_query("Does Python have a compiler", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.3912079334259033]: how a compiler works?
[3.357606887817383]: what is a compiler?
[3.284048080444336]: why do programmers need to use a compiler?


In [14]:
search_similar_query("I need to talk to the people behind this project", topk=3, docs=docs, doc_vecs=doc_vecs)

[3.5493767261505127]: it is nice to talk to you
[3.5471057891845703]: it is been so nice to talk to you
[3.546023368835449]: I want to talk with someone real


Lo probamos con algunos de los ejemplos que aparecen entre los datos de _test_ en el propio dataset.

In [15]:
for example in list(df_en[df_en["dataset"]=="test"].sample(10)["utterance"]):
    print(f"{example}")
    search_similar_query(example, 3, docs, doc_vecs)
    print()

you annoy me so much
[3.985309600830078]: you're so annoying
[3.9658010005950928]: you are boring me
[3.941418409347534]: How long you'll have me waiting

coercion in javascript
[3.6688308715820312]: how coercion works in javascript
[2.8955228328704834]: what is a javascript callback
[2.8280489444732666]: what is implicit coercion

I'm tracking the IP address of the attacker
[3.123244285583496]: I want the address of your office
[3.052868366241455]: I want to find the address of your company
[2.9944393634796143]: what's the address of your developer

who is in charge of your development
[3.4144842624664307]: show me were is the office of your developer
[3.37860369682312]: who is the boss of you
[3.3539764881134033]: what's the address of your developer

what are your hobbies?
[3.537806510925293]: do you have some hobby?
[3.4579293727874756]: what about your hobby
[3.4118196964263916]: what's your hobby

are you at work now?
[3.709036350250244]: are you working now?
[3.566328525543213]:

## `transformers` de Hugging Face

Te propongo realizar el siguiente ejercicio: replicar todo lo anterior sustituyendo los embeddings construidos con GloVe por un modelo de lenguaje más moderno, como BERT o [cualquiera de los modelos de lenguaje disponbles en `transformers`](https://huggingface.co/models). Para ello te recomiendo que revises el código del cuaderno anterior, [la documentación de la librería](https://huggingface.co/transformers/quicktour.html#under-the-hood-pretrained-models), y lo adaptes de manera que puedas:

- procesar un texto de entrada, tokenizarlo y construir el embedding
- procesar un texto de entrada, una colección de documentos en formato texto y la colección de embeddings, y calcular algún tipo de distancia de similitud para recuperar los documentos más similares.

## Quiero más modelos de lenguaje, ¿dónde busco?

Otra familia de modelos de lenguaje que funcionan muy bien útil para construir embeddings de textos heterogéneos es [Universal Sentence Encoder (USE)](https://tfhub.dev/google/collections/universal-sentence-encoder/1), de Google. 

Al contrario que los anteriores, estos modelos de lenguaje no están basado en la arquitectura de Transformer, sino que utiliza otro tipo de red neuronal llamado _Deep Averaging Network_ (DAN). 

Los modelos de USE están disponibles en TensorFlow Hub, tienen una API muy sencilla para generar los embeddings, y se distribuyen tanto [versiones específicas para el inglés](https://tfhub.dev/google/universal-sentence-encoder/4) como [versiones multilingües](https://tfhub.dev/google/universal-sentence-encoder-multilingual/3). Esta última versión es muy interesate y te puede dar mucho juego para repetir el ejercicio con los datasets del chatbot, mezclando textos en inglés y consultas en español, o viceversa.